In [ ]:
import os, dotenv, pathlib, typing, json

dotenv.load_dotenv()


# load all tar paths
TAR_PATHS = list(pathlib.Path("/mnt/data/datasets/downsampled_tars/").glob("*.tar"))
EMPTY_RESPONSE = {"intervals":[],"coverage_pct":0}
# define prompt
_FIND_MENUS_PROMPT = (
    "Identify time intervals (seconds) where large non-diegetic MENU overlays are visible "
    "(pause/settings/inventory/shop/map). Respond as: "
    '{"intervals":[{"start_sec":<float>,"end_sec":<float>,"confidence":<0..1>}],'
    '"coverage_pct":<0..1>}'
    "If you did not find any menus, respond with: "
    '{"intervals":[],"coverage_pct":0}'
    'Only JSON! And make sure that the intervals do not overlap, and are merged when necessary.'
)

# define csv path
CSV_PATH = "menu_intervals.csv"
# csv columns
CSV_COLUMNS = [
    "tar_name",
    "mp4_chunk_name",
    "menu_start_sec",
    "menu_end_sec",
    "menu_confidence",
    "menu_coverage_pct",
]

# -- 
print (f"Found {len(TAR_PATHS)} tar paths")
print (f"GEMINI_API_KEY Found: {os.getenv('GEMINI_API_KEY') is not None}")

Found 20228 tar paths
GEMINI_API_KEY Found: True


In [ ]:
def yield_mp4_bytes_from_tars(tar_paths: list[pathlib.Path]) -> typing.Generator[tuple[str, str, bytes], None, None]:
    """
    Yields a tuple of (tar_path, mp4_chunk_name, mp4_bytes) for each mp4 in tars
    """
    import tarfile

    for tar_path in tar_paths:
        with tarfile.open(tar_path) as tar:
            for member in tar.getmembers():
                if not member.name.endswith(".mp4"):
                    continue
                with tar.extractfile(member) as f:
                    yield (str(tar_path), member.name, f.read())

import google.genai as genai

CLIENT = genai.Client()
MODEL = 'gemini-2.5-flash-lite'


# async function that takes bytes from an mp4 and sends a query to gemini and parses the response into a list of csv rows per interval
def ask_gemini(tar_path: str, mp4_chunk_name: str, mp4_bytes: bytes) -> dict:
    global CLIENT, MODEL
    import google.genai.types as types
    
    contents = types.Content(parts=[
        types.Part(text=_FIND_MENUS_PROMPT),
        types.Part(inline_data=types.Blob(data=mp4_bytes, mime_type='video/mp4'))
    ])
    total_tokens = CLIENT.models.count_tokens(model=MODEL, contents=contents)

    print(f"Sending request to Gemini with {total_tokens} tokens")
    response: types.GenerateContentResponse = CLIENT.models.generate_content(
        model=MODEL,
        contents=contents,
    )

    output = response.candidates[0].content.parts[0].text
    try: 
        parsed = json.loads(output)
        parsed['error'] = None
        print(f"Parsed {len(parsed['intervals'])} intervals")
        return parsed
    except Exception as e:
        print (f"Error parsing output: {e} - output: {output} - for {tar_path} - {mp4_chunk_name}")
        err = EMPTY_RESPONSE
        err['error'] = str(e)
        return err

# function that writes csv rows to filepath
def write_csv_rows(intervals: dict, filepath: str, tar_name: str, mp4_chunk_name: str):
    import pandas as pd, os

    rows = [
        {
            "tar_name": tar_name,
            "mp4_chunk_name": mp4_chunk_name,
            "menu_start_sec": interval.get("start_sec", None),
            "menu_end_sec": interval.get("end_sec", None),
            "menu_confidence": interval.get("confidence", None),
            "menu_coverage_pct": interval.get("coverage_pct", None),
            "error": interval.get("error", None),
        }
        for interval in intervals.get('intervals', [{}])
    ]

    df = pd.DataFrame(rows)
    # append to csv instead of replacing it
    df.to_csv(filepath, mode='a', header=True, index=False)


for tar_path, mp4_chunk_name, mp4_bytes in yield_mp4_bytes_from_tars(TAR_PATHS):
    intervals = ask_gemini(tar_path, mp4_chunk_name, mp4_bytes)
    write_csv_rows(intervals, CSV_PATH, tar_path, mp4_chunk_name)




Sending request to Gemini with sdk_http_response=HttpResponse(
  headers=<dict len=11>
) total_tokens=17790 cached_content_token_count=None tokens
Parsed 0 intervals
Sending request to Gemini with sdk_http_response=HttpResponse(
  headers=<dict len=11>
) total_tokens=35490 cached_content_token_count=None tokens
Parsed 0 intervals
Sending request to Gemini with sdk_http_response=HttpResponse(
  headers=<dict len=11>
) total_tokens=53454 cached_content_token_count=None tokens
Parsed 0 intervals
Sending request to Gemini with sdk_http_response=HttpResponse(
  headers=<dict len=11>
) total_tokens=71154 cached_content_token_count=None tokens
Parsed 0 intervals
Sending request to Gemini with sdk_http_response=HttpResponse(
  headers=<dict len=11>
) total_tokens=88854 cached_content_token_count=None tokens
Parsed 0 intervals
Sending request to Gemini with sdk_http_response=HttpResponse(
  headers=<dict len=11>
) total_tokens=88584 cached_content_token_count=None tokens
Parsed 1 intervals
Send

KeyboardInterrupt: 